# Step 3: Analyze Video (REST API)

Demonstrate video analysis using the Azure Content Understanding REST API.
Transcription, scene descriptions, and keyframe extraction via direct HTTP calls.

## Load Configuration and Helpers

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

# Load env
demo_root = Path("..").resolve()
env_file = demo_root / "env" / ".env"
load_dotenv(dotenv_path=str(env_file))

# Add app to path
sys.path.insert(0, str(demo_root / "app"))
from notebook_helpers import analyze_video_rest, extract_descriptions_and_transcript

print(f"Demo root: {demo_root}")
print(f"✓ Configuration and helpers loaded")

## Configure Video Path

Set `VIDEO_PATH` to your local video file (< 1 min, < 200 MB, MP4/M4V/AVI/MKV/MOV/FLV/WMV).

In [ ]:
# CHANGE THIS TO YOUR VIDEO FILE
VIDEO_PATH = "../data/sample-video.mp4"  # <- Set your video path here

video_path = (demo_root / VIDEO_PATH).resolve()

if not video_path.exists():
    print(f"❌ Video file not found: {video_path}")
    print(f"\nPlease:")
    print(f"  1. Place a video file in {demo_root / 'data'} (or adjust VIDEO_PATH above)")
    print(f"  2. Re-run this cell")
    raise FileNotFoundError(str(video_path))

print(f"✓ Video file found: {video_path}")
print(f"  Size: {video_path.stat().st_size / 1024 / 1024:.1f} MB")

## Analyze Video via REST API

In [ ]:
# Get credentials and config
endpoint = os.getenv("AZURE_AI_ENDPOINT")
deployment_name = os.getenv("CU_MODEL_DEPLOYMENT_NAME")
credential = DefaultAzureCredential()

print(f"Video Analysis via REST API")
print(f"Endpoint: {endpoint}")
print(f"Deployment: {deployment_name}")
print()

# Analyze
result = analyze_video_rest(
    endpoint=endpoint,
    deployment_name=deployment_name,
    video_path=str(video_path),
    credential=credential,
)

print(f"Analysis complete!")

## Extract and Display Results

In [ ]:
# Extract key information
extracted = extract_descriptions_and_transcript(result)

print("=" * 60)
print("SCENE DESCRIPTIONS")
print("=" * 60)
for i, desc in enumerate(extracted["descriptions"], 1):
    print(f"\n[Scene {i}] {desc[:150]}..." if len(desc) > 150 else f"\n[Scene {i}] {desc}")

print("\n" + "=" * 60)
print("TRANSCRIPT")
print("=" * 60)
print(extracted["transcript"][:500] if extracted["transcript"] else "(No transcript extracted)")

print("\n" + "=" * 60)
print("KEYFRAMES")
print("=" * 60)
for kf in extracted["keyframes"][:5]:
    print(kf)

## Save Result

In [ ]:
import json

# Save full result
output_file = demo_root / "outputs" / "video_analysis_result_rest.json"
output_file.parent.mkdir(exist_ok=True)

with open(output_file, "w") as f:
    json.dump(result, f, indent=2, default=str)

print(f"✓ Result saved to {output_file}")